In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from bisect import bisect_left
from dateutil.relativedelta import relativedelta
from sklearn.model_selection import KFold
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl
import dgl.nn

# --- 데이터 전처리 함수 시작 ---
def calculate_age(born, trans_date):
    """생년월일과 거래 날짜를 이용하여 나이를 계산합니다."""
    return relativedelta(trans_date, born).years

def count_past_days_fast(df, days):
    """
    각 거래 시점을 기준으로 과거 'days' 기간 동안의 거래 횟수를 빠르게 계산합니다.
    df는 'cc_num'과 'trans_date_trans_time'으로 정렬되어 있어야 합니다.
    """
    results = []
    grouped = df.groupby('cc_num')

    for _, group in grouped:
        times = group['trans_date_trans_time'].tolist()
        counts = []
        for i, t in enumerate(times):
            start_time = t - pd.Timedelta(days=days)
            left_idx = bisect_left(times, start_time)
            right_idx = i
            counts.append(right_idx - left_idx)
        results.extend(counts)
    return results

def base_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    원본 데이터프레임에 기본적인 특징 엔지니어링을 적용합니다.
    로그 변환, 표준화, 시간 특징, 거래 빈도, 시간 간격, 나이 등을 포함합니다.
    그래프 구축에 필요한 ID 컬럼들은 유지합니다.
    """
    df_p = df.copy()

    # 불필요한 컬럼 제거 (그래프 구축에 사용되지 않는 원본 컬럼들)
    drop_cols_initial = ['Unnamed: 0', 'unix_time', 'trans_num', 'first', 'last', 'merchant',
                         'street', 'merch_lat', 'merch_long', 'city_pop', 'lat', 'long', 'zip']
    df_p = df_p.drop(columns=drop_cols_initial, errors='ignore')

    # 로그 변환 (amt)
    df_p['amt_log'] = np.log1p(df_p['amt'])

    # 표준화 (amt_log)
    scaler = StandardScaler()
    df_p['amt_log_std'] = scaler.fit_transform(df_p[['amt_log']])

    # datetime 변환 및 시간 관련 특징 추출
    df_p['datetime'] = pd.to_datetime(df_p['trans_date_trans_time'])
    df_p['hour'] = df_p['datetime'].dt.hour
    df_p['day_of_week'] = df_p['datetime'].dt.dayofweek # Monday=0, Sunday=6
    df_p['month'] = df_p['datetime'].dt.month # 1월=1, 12월=12

    # 시간 특징 (Sin/Cos 변환)
    df_p['trans_hour_sin'] = np.sin(2 * np.pi * df_p['hour'] / 24)
    df_p['trans_hour_cos'] = np.cos(2 * np.pi * df_p['hour'] / 24)
    df_p['is_night'] = df_p['hour'].isin([22, 23, 0, 1, 2, 3]).astype(int)

    # 고위험 시간대 특징
    high_risk_days = [2, 3, 4]  # 수, 목, 금
    high_risk_hours = list(range(0, 5)) + [21, 22, 23] # 0~4시, 21~23시
    df_p['is_high_risk_period'] = (
        df_p['day_of_week'].isin(high_risk_days) & df_p['hour'].isin(high_risk_hours)
    ).astype(int)

    # 거래 정렬 (cc_num별로 시간 순서)
    df_p['trans_date_trans_time'] = pd.to_datetime(df_p['trans_date_trans_time'])
    df_p = df_p.sort_values(['cc_num', 'trans_date_trans_time']).reset_index(drop=True)

    # 과거 N일간 거래 횟수
    df_p['cnt_1d'] = count_past_days_fast(df_p, 1)
    df_p['cnt_7d'] = count_past_days_fast(df_p, 7)
    df_p['cnt_30d'] = count_past_days_fast(df_p, 30)

    # 거래 시간 간격 특징
    df_p['next_trans_time'] = df_p.groupby('cc_num')['trans_date_trans_time'].shift(-1)
    df_p['time_since_last_trans'] = (df_p['trans_date_trans_time'] - df_p.groupby('cc_num')['trans_date_trans_time'].shift(1)).dt.total_seconds()
    df_p['time_until_next_trans'] = (df_p['next_trans_time'] - df_p['trans_date_trans_time']).dt.total_seconds()
    df_p[['time_since_last_trans', 'time_until_next_trans']] = df_p[['time_since_last_trans', 'time_until_next_trans']].fillna(0)

    # 나이 관련 특징
    df_p['dob'] = pd.to_datetime(df_p['dob'])
    df_p['age'] = df_p.apply(lambda x: calculate_age(x['dob'], x['trans_date_trans_time']), axis=1)
    df_p['age_group'] = (df_p['age'] // 10) * 10

    # 도시-주 결합 특징
    df_p['city_state'] = df_p['city'] + ', ' + df_p['state']

    # 범주형 변수 원-핫 인코딩
    df_p = pd.get_dummies(df_p, columns=['category'], prefix='category', drop_first=True)
    df_p = pd.get_dummies(df_p, columns=['gender'], prefix='gender', drop_first=True)

    # 최종 드롭 컬럼 (그래프 구축 후 드롭될 컬럼들 포함)
    # cc_num, trans_date_trans_time, city, state, job 등 그래프/타겟인코딩에 필요한 컬럼은 여기서 드롭하지 않음
    final_features_drop_cols = ['dob', 'next_trans_time', 'datetime', 'time_until_next_trans', 'amt_log', 'hour', 'amt']
    df_p = df_p.drop(columns=final_features_drop_cols, errors='ignore')

    return df_p

def preprocess_data(df: pd.DataFrame,
                    hour: str = 'sincos',
                    high_risk_period: bool = False,
                    age_group: bool = False
                    ) -> pd.DataFrame:
    """
    특정 전처리 옵션에 따라 컬럼을 선택하거나 제거합니다.
    Args:
        df: 입력 DataFrame.
        hour: 시간 전처리 방법 ('sincos' 또는 'is_night').
        high_risk_period: 고위험 시간대 컬럼 사용 여부.
        age_group: 연령대 그룹 컬럼 사용 여부.
    """
    df_p = df.copy()

    if hour == 'sincos':
        df_p = df_p.drop(columns=['is_night'], errors='ignore')
    elif hour == 'is_night':
        df_p = df_p.drop(columns=['trans_hour_sin','trans_hour_cos'], errors='ignore')

    if not high_risk_period:
        df_p = df_p.drop(columns=['is_high_risk_period'], errors='ignore')

    if age_group:
        df_p = df_p.drop(columns=['age'], errors='ignore')
    else:
        df_p = df_p.drop(columns=['age_group'], errors='ignore')

    return df_p


In [ ]:
# --- K-Fold 타겟 인코딩 스무딩 함수 시작 ---
def m_estimate_smoothing(mean, global_mean, count, m):
    """M-estimate 스무딩을 적용하여 평균을 계산합니다."""
    return (mean * count + global_mean * m) / (count + m)

def kfold_target_encoding(df, target_col, cat_col, n_splits=5, m_param=10):
    """
    K-Fold 교차 검증 방식의 타겟 인코딩을 M-estimate 스무딩과 함께 적용합니다.
    Args:
        df: 입력 데이터프레임.
        target_col: 타겟 컬럼 이름 (예: 'is_fraud').
        cat_col: 인코딩할 범주형 컬럼 이름.
        n_splits: K-Fold 분할 개수.
        m_param: M-estimate 스무딩 파라미터.
    Returns:
        인코딩된 컬럼이 추가된 DataFrame과 생성된 컬럼 이름들.
    """
    df_encoded = df.copy()
    global_mean = df[target_col].mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    oof_col = f'{cat_col}_enc_m_est_oof_m{m_param}'
    full_col = f'{cat_col}_enc_m_est_full_m{m_param}'
    df_encoded[oof_col] = np.nan

    # OOF (Out-Of-Fold) 방식 인코딩 (과적합 방지)
    for train_idx, valid_idx in kf.split(df):
        train_fold = df.iloc[train_idx]
        agg = train_fold.groupby(cat_col)[target_col].agg(['mean', 'count']).reset_index()
        agg['smoothed'] = m_estimate_smoothing(agg['mean'], global_mean, agg['count'], m_param)
        mapping = dict(zip(agg[cat_col], agg['smoothed']))
        df_encoded.loc[valid_idx, oof_col] = df.loc[valid_idx, cat_col].map(mapping).fillna(global_mean)

    # 전체 데이터 기준 인코딩 (최종 피처)
    agg_full = df.groupby(cat_col)[target_col].agg(['mean', 'count']).reset_index()
    agg_full['smoothed'] = m_estimate_smoothing(agg_full['mean'], global_mean, agg['count'], m_param)
    full_mapping = dict(zip(agg_full[cat_col], agg['smoothed']))
    df_encoded[full_col] = df[cat_col].map(full_mapping).fillna(global_mean)

    return df_encoded, oof_col, full_col

In [ ]:
# --- DGL 그래프 구성 함수 시작 ---

def build_heterogeneous_graph(df: pd.DataFrame):
    """
    전처리된 DataFrame으로부터 DGL 이종 그래프를 생성합니다.
    노드: transaction, credit_card
    엣지: uses_card, used_in, temporal_precedes,
          same_original_category, same_city_state
    """
    graph_data = {}

    # 1. 노드 ID 매핑
    # DataFrame의 인덱스를 transaction 노드 ID로 사용
    num_transactions = len(df)

    unique_cc_nums = df['cc_num'].unique()
    cc_num_to_idx = {cc_num: i for i, cc_num in enumerate(unique_cc_nums)}
    num_credit_cards = len(unique_cc_nums)

    # DGL 그래프의 num_nodes_dict 설정
    num_nodes_dict = {
        'transaction': num_transactions,
        'credit_card': num_credit_cards
    }

    # 2. 엣지 리스트 생성

    # Edge 1: (transaction) -- uses_card --> (credit_card)
    # Edge 2: (credit_card) -- used_in --> (transaction) (역방향)
    src_trans_uses_card = list(range(num_transactions))
    dst_cc_uses_card = df['cc_num'].map(cc_num_to_idx).tolist()
    graph_data[('transaction', 'uses_card', 'credit_card')] = (src_trans_uses_card, dst_cc_uses_card)
    graph_data[('credit_card', 'used_in', 'transaction')] = (dst_cc_uses_card, src_trans_uses_card) # 역방향

    # Edge 3: (transaction) -- temporal_precedes --> (transaction) (동일 카드 번호 내 시간 순서)
    temporal_src_nodes = []
    temporal_dst_nodes = []

    for i in range(1, num_transactions):
        if df.iloc[i]['cc_num'] == df.iloc[i-1]['cc_num']:
            temporal_src_nodes.append(df.index[i-1])
            temporal_dst_nodes.append(df.index[i])
    graph_data[('transaction', 'temporal_precedes', 'transaction')] = (temporal_src_nodes, temporal_dst_nodes)

    # Edge 4: (transaction) -- same_original_category --> (transaction) (동일 카테고리)
    category_cols = [col for col in df.columns if col.startswith('category_') and col != 'category_enc_m_est_full_m100' and col != 'category_enc_m_est_oof_m100']
    temp_original_category = df[category_cols].apply(lambda row: row.index[row.argmax()] if row.any() else None, axis=1)
    temp_original_category = temp_original_category.apply(lambda x: x.replace('category_', '') if x else None)

    same_cat_src_nodes = []
    same_cat_dst_nodes = []
    for cat, group in df.groupby(temp_original_category):
        if cat is None:
            continue
        trans_indices = group.index.tolist()
        if len(trans_indices) > 1:
            for i in range(len(trans_indices)):
                for j in range(i + 1, len(trans_indices)):
                    same_cat_src_nodes.append(trans_indices[i])
                    same_cat_dst_nodes.append(trans_indices[j])
                    same_cat_src_nodes.append(trans_indices[j])
                    same_cat_dst_nodes.append(trans_indices[i])
    graph_data[('transaction', 'same_original_category', 'transaction')] = (same_cat_src_nodes, same_cat_dst_nodes)

    # Edge 5: (transaction) -- same_city_state --> (transaction) (동일 city_state)
    same_cs_src_nodes = []
    same_cs_dst_nodes = []
    if 'city_state' in df.columns:
        for cs, group in df.groupby('city_state'):
            trans_indices = group.index.tolist()
            if len(trans_indices) > 1:
                for i in range(len(trans_indices)):
                    for j in range(i + 1, len(trans_indices)):
                        same_cs_src_nodes.append(trans_indices[i])
                        same_cs_dst_nodes.append(trans_indices[j])
                        same_cs_src_nodes.append(trans_indices[j])
                        same_cs_dst_nodes.append(trans_indices[i])
    else:
        print("Warning: 'city_state' column not found for 'same_city_state' edges. This relation will not be added.")
    graph_data[('transaction', 'same_city_state', 'transaction')] = (same_cs_src_nodes, same_cs_dst_nodes)


    # DGL 이종 그래프 생성
    g = dgl.heterograph(graph_data, num_nodes_dict=num_nodes_dict)

    # 노드 특징 할당
    transaction_features_cols = [
        'amt_log_std', 'day_of_week', 'month', 'trans_hour_sin', 'trans_hour_cos',
        'is_high_risk_period', 'cnt_1d', 'cnt_7d', 'cnt_30d', 'time_since_last_trans',
        'age_group',
        'category_food_dining', 'category_gas_transport', 'category_grocery_net',
        'category_grocery_pos', 'category_health_fitness', 'category_home',
        'category_kids_pets', 'category_misc_net', 'category_misc_pos',
        'category_personal_care', 'category_shopping_net', 'category_shopping_pos',
        'category_travel',
        'gender_M' # gender_F는 drop_first=True로 인해 생성되지 않음
    ]

    for col in df.columns:
        if '_enc_m_est_full_m' in col:
            if col not in transaction_features_cols:
                transaction_features_cols.append(col)

    transaction_features_cols = [col for col in transaction_features_cols if col in df.columns and col != 'job']

    missing_cols = [col for col in transaction_features_cols if col not in df.columns]
    if missing_cols:
        print(f"Warning: The following feature columns are missing from the DataFrame: {missing_cols}")

    initial_transaction_features = torch.tensor(
        df[transaction_features_cols].fillna(0).values, dtype=torch.float32
    )
    g.nodes['transaction'].data['feat'] = initial_transaction_features

    target_labels = torch.tensor(df['is_fraud'].values, dtype=torch.float32)

    return g, initial_transaction_features, target_labels


In [ ]:
# --- GNN-CL 모델 아키텍처 시작 ---
class GNNLayer(nn.Module):
    """
    GNN-CL 모델의 GNN 레이어입니다.
    DGL 그래프 객체를 입력받아 메시지 전달을 수행합니다.
    """
    def __init__(self, in_features, out_features):
        super(GNNLayer, self).__init__()
        self.in_features = in_features
        self.out_features = out_features

        # Purification Network: 노드 특징을 정화하여 전달할 메시지를 제어
        self.mlp_purifier = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.ReLU(),
            nn.Linear(in_features, in_features),
            nn.Sigmoid() # 0-1 사이의 가중치를 생성하여 메시지를 필터링
        )

        # 각 관계 타입별 GraphConv 레이어
        # 여기에 정의된 관계는 build_heterogeneous_graph에서 생성된 관계와 일치해야 합니다.
        self.conv_layers = nn.ModuleDict({
            'uses_card': dgl.nn.GraphConv(in_features, out_features),
            'used_in': dgl.nn.GraphConv(in_features, out_features),
            'temporal_precedes': dgl.nn.GraphConv(in_features, out_features),
            'same_original_category': dgl.nn.GraphConv(in_features, out_features),
            'same_city_state': dgl.nn.GraphConv(in_features, out_features),
        })

        # Relationship Summarizer: 모든 관계에서 집계된 특징을 결합하여 최종 노드 임베딩 생성
        # (원본 특징 + 관계별 집계 특징들) -> out_features
        self.final_summarizer_linear = nn.Linear(in_features + out_features * len(self.conv_layers), out_features)

    def forward(self, g, h_prev):
        with g.local_scope():
            # 노드 특징 할당
            g.nodes['transaction'].data['h'] = h_prev

            # Purification Network 적용: 각 노드가 메시지를 보낼 때 얼마나 중요한지 결정
            purification_weights = self.mlp_purifier(h_prev)

            aggregated_h_per_relation = []

            # 각 관계별로 메시지 전달 및 집계
            for etype, conv_layer in self.conv_layers.items():
                src_type, _, dst_type = g.canonical_etypes[g.get_etype_id(etype)]

                sg = g[src_type, etype, dst_type]

                h_src = sg.srcdata['h'] if src_type == 'transaction' else torch.zeros(sg.num_src_nodes(), self.in_features, device=h_prev.device)

                if src_type == 'transaction':
                    h_src_purified = h_src * purification_weights[sg.srcdata[dgl.NID]]
                else:
                    h_src_purified = h_src

                if dst_type == 'transaction':
                    if etype in ['uses_card', 'used_in', 'temporal_precedes', 'same_original_category', 'same_city_state']:
                        # 모든 관련 엣지 타입을 처리 (src와 dst가 transaction인 경우와, credit_card -> transaction 경우)
                        # 현재는 credit_card 노드의 특징이 없으므로 src_type이 transaction인 경우만 h_prev를 사용
                        if src_type == 'transaction': # transaction-to-transaction 관계
                             h_rel = conv_layer(sg, h_prev)
                             aggregated_h_per_relation.append(h_rel)
                        elif src_type == 'credit_card': # credit_card -> transaction 관계
                            # credit_card 노드의 특징이 있다면, 그 특징을 사용해야 합니다.
                            # 현재는 credit_card 노드에 특징이 할당되지 않았으므로, 0 벡터를 사용하거나 이 부분을 스킵.
                            # 여기서는 0 벡터를 가정하고 진행합니다. (실제 모델링 시에는 의미 있는 특징이 필요)
                            # h_rel = conv_layer(sg, torch.zeros(sg.num_src_nodes(), self.in_features, device=h_prev.device))
                            # aggregated_h_per_relation.append(h_rel)
                            pass # 현재는 credit_card 노드의 특징이 없으므로 스킵. 필요시 구현

            if len(aggregated_h_per_relation) == 0:
                # 모든 aggregated_h_per_relation의 크기를 통일 (out_features)
                # 빈 리스트일 경우, self.conv_layers의 수에 맞춰 0 벡터를 생성하여 concat
                concatenated_h = torch.cat([h_prev] + [torch.zeros(h_prev.size(0), self.out_features, device=h_prev.device)] * len(self.conv_layers), dim=1)
            else:
                concatenated_h = torch.cat([h_prev] + aggregated_h_per_relation, dim=1)

            h_current = F.relu(self.final_summarizer_linear(concatenated_h))
            return h_current

In [ ]:
class CNNLayer(nn.Module):
    """
    GNN-CL 모델의 CNN 레이어입니다.
    GNN 출력을 시퀀스로 보고 특징을 추출합니다.
    """
    def __init__(self, in_features_from_gnn, out_channels, kernel_size):
        super(CNNLayer, self).__init__()
        # Conv1d는 (batch_size, in_channels, sequence_length) 입력을 기대
        self.conv1d = nn.Conv1d(in_features_from_gnn, out_channels, kernel_size, padding=kernel_size//2)
        # MaxPool1d는 (batch_size, channels, sequence_length) 입력을 기대
        self.pool = nn.MaxPool1d(kernel_size)
        self.linear_transform = nn.Linear(out_channels, out_channels)

    def forward(self, x):
        # 입력 x (batch_size, num_nodes, features) -> (batch_size, features, num_nodes)로 permute
        x = x.permute(0, 2, 1)
        x = F.relu(self.conv1d(x))
        x = self.pool(x)
        # 풀링 후 남은 차원(num_nodes)에 대해 평균을 취하여 각 배치 요소당 하나의 벡터를 얻음
        x = x.mean(dim=2)
        x = self.linear_transform(x)
        return x

In [ ]:
class LSTMLayer(nn.Module):
    """
    GNN-CL 모델의 LSTM 레이어입니다.
    CNN 출력을 시퀀스로 보고 최종 예측을 수행합니다.
    노드 레벨 분류를 위해 (num_nodes, 1) 출력을 반환하도록 수정되었습니다.
    """
    def __init__(self, input_size, hidden_size, num_layers=1, bidirectional=True):
        super(LSTMLayer, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        # LSTM은 (batch_size, seq_len, input_size) 입력을 기대
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=bidirectional)

        # 양방향일 경우 hidden_size * 2, 아닐 경우 hidden_size
        output_linear_input_size = hidden_size * 2 if bidirectional else hidden_size
        # 노드 레벨 예측 (각 노드에 대해 1개의 예측값)
        self.output_linear = nn.Linear(output_linear_input_size, 1)

    def forward(self, x):
        x = x.unsqueeze(1)
        output, _ = self.lstm(x)

        output_combined = self.output_linear(output.squeeze(1))

        # Sigmoid를 통해 0과 1 사이의 확률값 반환
        return torch.sigmoid(output_combined)


In [ ]:
class GNN_CL_Model(nn.Module):
    """
    GNN-CL 전체 모델입니다.
    GNN, CNN, LSTM 레이어를 순차적으로 결합합니다.
    """
    def __init__(self, node_features_dim, gnn_out_features,
                cnn_out_channels, cnn_kernel_size, lstm_hidden_size):
        super(GNN_CL_Model, self).__init__()
        self.gnn_layer = GNNLayer(node_features_dim, gnn_out_features)
        self.cnn_layer = CNNLayer(gnn_out_features, cnn_out_channels, cnn_kernel_size)
        self.lstm_layer = LSTMLayer(cnn_out_channels, lstm_hidden_size)

    def forward(self, g, initial_node_features, labels=None, epoch_info=None):
        # GNN Layer: 노드 특징과 그래프 구조를 통해 임베딩 업데이트
        h_gnn = self.gnn_layer(g, initial_node_features)

        # CNN Layer: GNN 출력을 (1, num_nodes, gnn_out_features) 형태로 변환하여 입력 (배치 차원 추가)
        h_gnn_batch = h_gnn.unsqueeze(0)
        h_cnn = self.cnn_layer(h_gnn_batch) # h_cnn.shape: (1, cnn_out_channels)

        # LSTM Layer: CNN 출력을 받아 최종 예측 수행
        output = self.lstm_layer(h_cnn) # output.shape: (1, 1) -> 하나의 스칼라 값으로 압축
        return output



In [ ]:

# --- 메인 실행 흐름 ---

if __name__ == "__main__":
    # --- 1. 데이터 로드: fraudTrain.csv (학습 데이터) 및 fraudTest.csv (테스트 데이터) ---
    print("--- 1. Loading Data: fraudTrain.csv (Training) and fraudTest.csv (Test) ---")

    raw_df = pd.read_csv('fraudTrain.csv')
    print(f"Loaded fraudTrain.csv with {len(raw_df)} samples.")
    print(f"Fraud ratio in fraudTrain.csv: {raw_df['is_fraud'].mean():.2f}")

    test_df = pd.read_csv('fraudTest.csv')
    print(f"Loaded fraudTest.csv with {len(test_df)} samples. (This will be used for evaluation later)")

    print("-" * 50)

    # --- 2. CTGAN을 통한 데이터 증강 (실행하려면 아래 주석을 해제하고 'sdv' 라이브러리 설치 필요) ---
    print("--- 2. CTGAN Data Augmentation (Requires 'sdv' library installation) ---")

    # # 'sdv' 라이브러리가 설치되어 있지 않다면, 먼저 설치하세요: pip install sdv
    # from sdv.single_table.ctgan import CTGAN

    # augmented_data_size = 55000
    # epochs_ctgan = 300

    # # CTGAN에 사용할 범주형 컬럼들 정의 (예: 원본 데이터에 존재하는 컬럼)
    # categorical_features_for_ctgan = ['category', 'city', 'state', 'cc_num', 'is_fraud', 'job', 'gender']

    # print(f"Initializing CTGAN with {epochs_ctgan} epochs...")
    # model_ctgan = CTGAN(epochs=epochs_ctgan)
    # print("Fitting CTGAN model to raw_df (this may take time)...")
    # model_ctgan.fit(raw_df, primary_key=None, metadata=None, categorical_features=categorical_features_for_ctgan)
    # print("CTGAN model fitted. Generating synthetic data...")

    # synthetic_data = model_ctgan.sample(augmented_data_size)
    # print(f"Generated {len(synthetic_data)} synthetic samples.")

    # combined_df = pd.concat([raw_df, synthetic_data], ignore_index=True)
    # print(f"Combined DataFrame size after CTGAN: {len(raw_df)} + {augmented_data_size} = {len(combined_df)}")

    combined_df = raw_df.copy()
    print("CTGAN execution skipped (uncomment code to enable). Using original data for next steps.")
    print("-" * 50)

    # --- 3. base_df 전처리 실행 ---
    print("--- 3. Running base_df Preprocessing ---")
    processed_df = base_df(combined_df)
    print(f"DataFrame after base_df: {processed_df.shape}")
    print("-" * 50)

    # --- 4. preprocess_data 선택 전처리 실행 ---
    print("--- 4. Running preprocess_data Selective Preprocessing ---")
    final_processed_df = preprocess_data(processed_df, hour='sincos', high_risk_period=True, age_group=True)
    print(f"DataFrame after preprocess_data: {final_processed_df.shape}")
    print("Columns after base_df and preprocess_data:", final_processed_df.columns.tolist())
    print("-" * 50)

    # --- 5. K-Fold 타겟 인코딩 (스무딩 적용) ---
    print("--- 5. K-Fold Target Encoding with M-Estimate Smoothing ---")

    categorical_cols_for_smoothing = ['city_state']
    m_param_smoothing = 100

    df_with_smoothed_features = final_processed_df.copy()

    for cat_col in categorical_cols_for_smoothing:
        if cat_col in df_with_smoothed_features.columns:
            df_with_smoothed_features, oof_col, full_col = kfold_target_encoding(
                df_with_smoothed_features, 'is_fraud', cat_col, m_param=m_param_smoothing
            )
            print(f"  - Added smoothed features for '{cat_col}': {full_col} (OOF: {oof_col})")
        else:
            print(f"  - Warning: Column '{cat_col}' not found in DataFrame for smoothing. Skipping.")

    if 'job' in df_with_smoothed_features.columns:
        df_with_smoothed_features, job_oof_col, job_full_col = kfold_target_encoding(
            df_with_smoothed_features, 'is_fraud', 'job', m_param=m_param_smoothing
        )
        print(f"  - Added smoothed features for 'job': {job_full_col} (OOF: {job_oof_col})")
    else:
        print("  - Warning: 'job' column not found in DataFrame for smoothing. Skipping.")

    print(f"DataFrame after K-Fold Target Encoding: {df_with_smoothed_features.shape}")
    print("New smoothed columns:", [col for col in df_with_smoothed_features.columns if '_enc_m_est_' in col])
    print("-" * 50)

    # --- 6. DGL 그래프 구축 및 GNN-CL 모델 입력 데이터 준비 ---
    print("--- 6. Building DGL Heterogeneous Graph and Preparing GNN-CL Model Input ---")

    g, initial_features_for_gnn, labels_for_gnn = build_heterogeneous_graph(df_with_smoothed_features)

    print(f"DGL Graph: {g}")
    print(f"Initial Transaction Node Features shape: {initial_features_for_gnn.shape}")
    print(f"Target Labels shape: {labels_for_gnn.shape}")

    # 그래프 구축에 사용된 원본 ID 및 구조 정의 컬럼들은 이제 최종 학습 데이터에서 드롭
    final_features_for_model = df_with_smoothed_features.drop(columns=['cc_num', 'city', 'state', 'trans_date_trans_time', 'job'], errors='ignore')
    print(f"Final DataFrame shape for non-graph features (after dropping structural columns): {final_features_for_model.shape}")
    print("-" * 50)

    # --- 7. GNN-CL 모델 학습 (개념적) ---
    print("--- 7. Starting GNN-CL Training Loop (Conceptual) ---")

    # 하이퍼파라미터
    node_features_dim = initial_features_for_gnn.shape[1]
    gnn_out_features = 64
    cnn_out_channels = 32
    cnn_kernel_size = 3
    lstm_hidden_size = 16

    learning_rate = 0.01
    epochs = 10

    model = GNN_CL_Model(node_features_dim, gnn_out_features,
                         cnn_out_channels, cnn_kernel_size, lstm_hidden_size)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.BCELoss()

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        epoch_info = {'is_training': True, 'epoch': epoch}
        output = model(g, initial_features_for_gnn, labels=labels_for_gnn, epoch_info=epoch_info)

        loss = criterion(output.squeeze(), labels_for_gnn)

        loss.backward()
        optimizer.step()

        if (epoch + 1) % 1 == 0:
            with torch.no_grad():
                predicted = (output.squeeze() > 0.5).float()
                accuracy = (predicted == labels_for_gnn).float().mean()
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}, Accuracy: {accuracy.item():.4f}")

    print("--- GNN-CL Training Finished ---")